# Week 6 controlled gate diagnostics

This notebook runs only the predeclared target audit and fixed-sample controls. It does not inspect OOD data or reuse production checkpoint state. Attach the private dataset containing `chronopde.h5`, enable a T4 GPU, enable Internet, then choose **Save Version → Save & Run All**. The final cell packages partial artifacts even when the scientific gate fails.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import FileLink, display

IMPLEMENTATION_COMMIT = '80bafb5f09a2d924535f1530ed104bda8d62a714'
REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
REPOSITORY = Path('/kaggle/working/Chrono_pde')
OUTPUT = REPOSITORY / 'artifacts/diagnostics/week6'
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = (
    '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
)

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Notebook settings'
print('GPU:', torch.cuda.get_device_name(0))
print('Visible GPU count:', torch.cuda.device_count())


In [ ]:
input_root = Path('/kaggle/input')
data_candidates = [
    path
    for path in input_root.rglob('chronopde.h5')
    if path.is_file() and path.stat().st_size == EXPECTED_DATA_SIZE
]
assert data_candidates, 'Attach the private dataset containing chronopde.h5'
DATA = data_candidates[0]
digest = hashlib.sha256()
with DATA.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
actual_hash = digest.hexdigest()
assert actual_hash == EXPECTED_DATA_SHA256, (
    f'Dataset hash mismatch: expected {EXPECTED_DATA_SHA256}, got {actual_hash}'
)
print('Verified dataset:', DATA)


In [ ]:
if REPOSITORY.exists():
    shutil.rmtree(REPOSITORY)
subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(
    ['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT],
    check=True,
)
os.chdir(REPOSITORY)
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--ignore-requires-python',
        '-e',
        '.[dev]',
    ],
    check=True,
)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == IMPLEMENTATION_COMMIT
print('Pinned commit:', actual_commit)


In [ ]:
command = [
    sys.executable,
    'scripts/diagnose_continuous.py',
    '--config',
    'configs/project.yaml',
    '--data-path',
    str(DATA),
    '--device',
    'cuda',
]
print('Launching:', ' '.join(command))
diagnostic = subprocess.run(command, check=False)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_return_code.json').write_text(
    json.dumps({'return_code': diagnostic.returncode}, indent=2) + '\n'
)
print('Diagnostic return code:', diagnostic.returncode)


In [ ]:
summary_path = OUTPUT / 'suite_summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    print('Routing decision:', summary['route'])
else:
    print('No suite summary was produced; package partial diagnostics for debugging.')


In [ ]:
provenance = REPOSITORY / 'reports/diagnostics/week6'
if provenance.is_dir():
    shutil.copytree(provenance, OUTPUT / 'prior_evidence', dirs_exist_ok=True)
package = shutil.make_archive(
    '/kaggle/working/chronopde_week6_gate_diagnostics',
    'zip',
    root_dir=OUTPUT,
)
print('Downloadable diagnostic package:', package)
display(FileLink(package))
